<a href="https://colab.research.google.com/github/DudeUnoob/Qwen3.8-27B-PEFT-medical/blob/main/Qwen3_5_9B_Clean_Steering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3.5-9B — neuron steering experiment

Steering with an individual neuron i saw from twitter as an experiment vs baseline

In [1]:
%pip install -U "transformers>=5.0.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 12.7 MB/s  0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2


In [8]:
import time
from contextlib import contextmanager

import torch
from transformers import pipeline

MODEL_ID = "Qwen/Qwen3.5-9B"
LAYER = 17
NEURON = 9398
STRENGTH = 0.8
MAX_NEW_TOKENS = 128

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is recommended for this 9B model. Enable GPU runtime first.")

print("GPU:", torch.cuda.get_device_name(0))

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    model_kwargs={"attn_implementation": "sdpa", "use_kernels": False},
)
model = pipe.model.eval()
tokenizer = pipe.tokenizer
print("Loaded:", type(model).__name__)

GPU: NVIDIA A100-SXM4-40GB


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Loaded: Qwen3_5ForConditionalGeneration


In [9]:
def find_transformer_layers(root):
    """Return the first transformer block ModuleList found in a model."""
    candidates = [
        (name, module)
        for name, module in root.named_modules()
        if isinstance(module, torch.nn.ModuleList)
        and name.split(".")[-1] == "layers"
    ]
    if not candidates:
        raise RuntimeError("Could not find transformer layers on the loaded model.")
    return max(candidates, key=lambda item: len(item[1]))[1]


layers = find_transformer_layers(model)
if not 0 <= LAYER < len(layers):
    raise IndexError(f"Layer {LAYER} out of range (total: {len(layers)}).")

mlp = layers[LAYER].mlp
if not hasattr(mlp, "down_proj"):
    raise TypeError(f"Layer {LAYER} MLP has no down_proj; check the architecture.")

width = mlp.down_proj.in_features
if not 0 <= NEURON < width:
    raise IndexError(f"Neuron {NEURON} out of range (MLP width: {width}).")

print(f"Ready: {len(layers)} layers; layer {LAYER}; MLP width {width}; neuron {NEURON}.")

Ready: 32 layers; layer 17; MLP width 12288; neuron 9398.


In [10]:
def generate_answer(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS):
    """Return only freshly generated text, excluding the user prompt."""
    messages = [{"role": "user", "content": prompt}]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    )

    if isinstance(encoded, torch.Tensor):
        inputs = {"input_ids": encoded}
    else:
        inputs = dict(encoded)

    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    prompt_length = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    text = tokenizer.decode(generated[0, prompt_length:], skip_special_tokens=True).strip()
    return {"text": text, "seconds": time.perf_counter() - t0,
            "tokens": generated.shape[-1] - prompt_length}

In [16]:
@contextmanager
def steer_neuron(layer: int, neuron: int, strength: float):
    """Temporarily add a constant to one MLP intermediate activation."""
    if not 0 <= layer < len(layers):
        raise IndexError(f"Invalid layer {layer}; model has {len(layers)} blocks.")
    down_proj = layers[layer].mlp.down_proj
    if not 0 <= neuron < down_proj.in_features:
        raise IndexError(f"Invalid neuron {neuron}; width is {down_proj.in_features}.")

    stats = {"hook_calls": 0, "modified_elements": 0}

    def pre_hook(module, args):
        x = args[0]
        changed = x.clone()
        changed[..., neuron] = changed[..., neuron] + strength
        stats["hook_calls"] += 1
        stats["modified_elements"] += changed[..., neuron].numel()
        return (changed, *args[1:])

    handle = down_proj.register_forward_pre_hook(pre_hook)
    try:
        yield stats
    finally:
        handle.remove()

In [17]:
def compare_steering(prompt, layer=LAYER, neuron=NEURON,
                     strength=STRENGTH, max_new_tokens=MAX_NEW_TOKENS):
    baseline = generate_answer(prompt, max_new_tokens)
    with steer_neuron(layer, neuron, strength) as stats:
        steered = generate_answer(prompt, max_new_tokens)

    if stats["hook_calls"] == 0:
        raise RuntimeError("Steering hook never ran; the selected down_proj was bypassed.")

    print(f"Prompt: {prompt}")
    print(f"Target: L{layer} N{neuron}, activation offset {strength:+g}")
    print(f"Hook executed: {stats['hook_calls']} times;"
          f" modified {stats['modified_elements']} activation values")
    print(f"\nBASELINE ({baseline['seconds']:.2f}s, {baseline['tokens']} tokens):\n{baseline['text']}")
    print(f"\nSTEERED ({steered['seconds']:.2f}s, {steered['tokens']} tokens):\n{steered['text']}")
    print("\nOutput changed:", baseline["text"] != steered["text"])
    return {"baseline": baseline, "steered": steered, "hook_stats": dict(stats)}

In [24]:
PROMPT = "Explain what a neural network is in one sentence."
result = compare_steering(PROMPT, layer=17, neuron=9398, strength=0.8)

Prompt: Explain what a neural network is in one sentence.
Target: L17 N9398, activation offset +0.8
Hook executed: 32 times; modified 54 activation values

BASELINE (2.41s, 39 tokens):
A neural network is a computational model inspired by the human brain that consists of interconnected layers of nodes which process data through weighted connections and activation functions to learn complex patterns and make predictions.

STEERED (1.96s, 32 tokens):
A neural network is a computational model inspired by the human brain that consists of interconnected layers of nodes which process data to recognize patterns and make decisions.

Output changed: True
